In [1]:
import pandas as pd
import os

In [ ]:
def add_gait_count(folder_path):
    print("Working on", folder_path)   
    df = pd.read_csv(os.path.join(folder_path, "merged.csv"))

    step_count = 0
    step_counts = []

    # Loop through each row and update the step count
    for is_step in df['insoles_LeftFoot_is_step']:
        if is_step:  # Increment the step count at each TRUE
            step_count += 1
        step_counts.append(step_count)

    df['stepcount'] = step_counts

    # Group by 'step' and filter out groups with more than one unique 'walk_mode' label
    filtered_df = (
        df.groupby('stepcount')
        .filter(lambda x: x['walk_mode'].nunique() == 1)
    )

    # Loop through all gait cycles and filter out the ones that have a length outside of the acceptable range
    filtered_df = (
        filtered_df.groupby('stepcount')
        .filter(lambda x: 41 <= len(x) <= 85)
    )

    # Reassign the `stepcount` column to be continuous and start from 1
    filtered_df['stepcount'] = pd.factorize(filtered_df['stepcount'])[0] + 1
    filtered_df.to_csv(os.path.join(folder_path, "merged_gait_count_annotations.csv"), index=False)

In [ ]:
dataset_path = "data_set"

print("Adding gait count according to the provided annotations...")
for course_folder in os.listdir(dataset_path):
    course_folder_path = os.path.join(dataset_path, course_folder)
    if os.path.isdir(course_folder_path):
        for subfolder in os.listdir(course_folder_path):
            subfolder_path = os.path.join(course_folder_path, subfolder)
            if os.path.isdir(subfolder_path):
                add_gait_count(subfolder_path)
print("Gait count added successfully")